# 05 — Run `noPE` (baseline: no positional encoding)

Trains `configs/GPS/pcqm4m-subset-GPS-noPE.yaml` for the seeds you list below.

Identical GPS-small backbone as the other stability baselines (LapPE / SignNet-MLP / SignNet-DeepSets / L-HKS); the only change is that `node_encoder_name` is plain `Atom` and there is no `posenc_*:` block. This gives the PE-stability experiments a "no-PE" reference point — Exp 1 (edge removal) still measures a signal (pure MPNN sensitivity to topology edits), while Exp 2 (eigenvector rotation) is trivially zero for this method.

**Prerequisite:** `MyDrive/datasets/pcqm4m-v2/processed/` must contain the OGB-processed PCQM4Mv2 `.pt` file(s). Either run `00_setup.ipynb` once, or upload/sync your existing `datasets/` tree to Drive at `MyDrive/datasets/`.

**Drive (data) account:** **`znidar.mark@gmail.com`** — authorize this account in the Drive popup so `datasets/` and `results_pcqm4m_subset/` resolve correctly. Colab may be logged in as a different account (e.g. one with Pro compute); only the Drive popup selects where data goes.

**Time budget:** ~1.5 h / seed on A100, ~4 h / seed on T4 (slightly faster than LHKS since there is no PE encoder forward).

## 1. Configuration — edit the seed list

In [ ]:
# =========================================================
# EDIT: which seeds to run
# =========================================================
SEEDS = [1]
# =========================================================

GITHUB_REPO     = "https://github.com/mark-znidar/gdl__2.git"
DRIVE_WORKSPACE = "/content/drive/MyDrive"
CFG             = "configs/GPS/pcqm4m-subset-GPS-noPE.yaml"
OUTSUBDIR       = "stability_baselines/nope"

DRIVE_RESULTS  = f"{DRIVE_WORKSPACE}/results_pcqm4m_subset"
DRIVE_DATASETS = f"{DRIVE_WORKSPACE}/datasets"
print("Will run noPE for seeds:", SEEDS)

## 2. GPU check

Prefer A100 or V100; T4 works but is slower.

In [ ]:
!nvidia-smi | head -20

## 3. Mount Drive

In [ ]:
from google.colab import drive
import os, shutil, time

MOUNTPOINT = "/content/drive"
MOUNT_TIMEOUT_MS = 180_000  # 3 min: auth + FUSE (slow network / large Drive)

def _clean_mountpoint():
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if os.path.islink(MOUNTPOINT):
        os.remove(MOUNTPOINT)
    elif os.path.isdir(MOUNTPOINT) and os.listdir(MOUNTPOINT):
        shutil.rmtree(MOUNTPOINT)
    os.makedirs(MOUNTPOINT, exist_ok=True)

def _mount(force_remount=False):
    drive.mount(
        MOUNTPOINT,
        timeout_ms=MOUNT_TIMEOUT_MS,
        force_remount=force_remount,
    )

_clean_mountpoint()
try:
    _mount()
except ValueError as e:
    if "mount failed" not in str(e).lower():
        raise
    print(
        "=== Google Drive: mount failed ===\n"
        "1) Re-run this cell; complete the auth popup (Allow).\n"
        "2) Allow pop-ups for Colab and Google accounts.\n"
        "3) Sign in with the account that has MyDrive/datasets.\n"
        "4) Runtime -> Restart runtime, then run this section again.\n"
        "5) https://research.google.com/colaboratory/faq.html#drive-timeout\n\n"
        "Retrying once in 5s with force_remount=True ...\n",
        flush=True,
    )
    time.sleep(5)
    _clean_mountpoint()
    _mount(force_remount=True)


## 4. Clone repo + install deps + symlink to Drive

In [ ]:
import os
%cd /content
if not os.path.isdir("gdl__2"):
    !git clone {GITHUB_REPO}
else:
    %cd gdl__2 && !git pull && %cd ..
%cd gdl__2
!bash run_colab/setup.sh
!rm -rf results_pcqm4m_subset datasets
!ln -sfn {DRIVE_RESULTS}  results_pcqm4m_subset
!ln -sfn {DRIVE_DATASETS} datasets
!ls -la results_pcqm4m_subset datasets

## 5. WandB login

In [ ]:
import wandb
wandb.login()

## 6. Sanity check — dataset is on Drive

Checks that `MyDrive/datasets/pcqm4m-v2/processed/` exists and contains a **large** `.pt` (OGB processed graphs). If this fails, run `00_setup.ipynb` or copy your local `datasets/pcqm4m-v2/` into that Drive path.

In [ ]:
from pathlib import Path

proc = Path(DRIVE_DATASETS) / "pcqm4m-v2" / "processed"
assert proc.is_dir(), (
    f"Missing {proc}. Put your OGB tree under MyDrive/datasets/ or run run_colab/00_setup.ipynb."
)
pts = sorted(p for p in proc.glob("*.pt") if p.is_file())
assert pts, f"No .pt files under {proc}"
big = [p for p in pts if p.stat().st_size > 500_000_000]
assert big, (
    "No large processed .pt found (expected a multi-GB OGB file such as "
    "geometric_data_processed.pt). Files here: "
    + ", ".join(f"{p.name} ({p.stat().st_size/1e6:.1f} MB)" for p in pts)
)
for p in big:
    print(f"OK  {p.name}  {p.stat().st_size / 1e9:.1f} GB")

## 7. Run training for each seed

Loops sequentially, skips seeds that already have a checkpoint on Drive. Safe to re-run after a disconnect.

In [ ]:
import os, glob, datetime, subprocess

OUTBASE = f"results_pcqm4m_subset/{OUTSUBDIR}"

for seed in SEEDS:
    outdir = f"{OUTBASE}/seed{seed}"
    if glob.glob(f"{outdir}/*/ckpt/*.ckpt"):
        print(f"[skip] noPE seed={seed} — checkpoint already exists at {outdir}")
        continue
    os.makedirs(outdir, exist_ok=True)
    print(f"\n{'='*60}")
    print(f">>> noPE seed={seed}  start: {datetime.datetime.now().isoformat(timespec='seconds')}")
    print(f">>> cfg:    {CFG}")
    print(f">>> outdir: {outdir}")
    print(f"{'='*60}", flush=True)
    ret = subprocess.call(["python", "main.py", "--cfg", CFG,
                           "--repeat", "1", "seed", str(seed),
                           "out_dir", outdir])
    status = "OK" if ret == 0 else f"FAILED (exit {ret})"
    print(f">>> noPE seed={seed}  done:  {datetime.datetime.now().isoformat(timespec='seconds')}  [{status}]")
    if ret != 0:
        raise RuntimeError(f"Training failed for noPE seed={seed}")
print("\n=== All noPE runs complete ===")

## 8. Verify checkpoints on Drive

In [ ]:
import subprocess
out = subprocess.check_output(
    ["find", f"{DRIVE_RESULTS}/{OUTSUBDIR}", "-name", "*.ckpt"], text=True)
paths = sorted(out.strip().split('\n')) if out.strip() else []
print(f"noPE has {len(paths)} checkpoint(s) on Drive:")
for p in paths:
    sz = os.path.getsize(p) / 1e6
    print(f"  {sz:6.1f} MB  {p}")